# 投机解码

> 让一个 0.5B 的小模型，给 70B 的大模型提速两倍——而且最终输出的分布和直接用大模型完全相同。这听起来像作弊，却是 2023 年以来推理侧最重要的优化之一。
>
> 它成立的原因藏在上一章的结论里：Decode 是串行的——每个 Token 都要等上一个 Token 确认后才能开始算。串行才是瓶颈，而不是单步的计算量。
>
> 本章介绍投机解码的四块内容：
>
> 1. **串行瓶颈**：为什么提速单步救不了自回归。
> 2. **Draft 与 Verify**：小模型猜、大模型一次验的核心流程。
> 3. **接受与校正**：min(1, p/q) 接受准则和校正分布——保证分布不变的关键。
> 4. **收益公式**：接受率、猜测长度、draft 开销如何共同决定加速比。

## 1. 自回归的串行瓶颈

普通 Decode 的循环是这样的：

```text
Target forward -> token1
Target forward -> token2   <- 必须等 token1 确定
Target forward -> token3   <- 必须等 token2 确定
Target forward -> token4
```

量化让每一步变便宜了，KV Cache 让每一步不用重算历史——但「每步只能确认一个 Token」这件事没有变。四个 Token 就是四次串行的大模型前向，一步都省不掉。

换个角度想：模型一次 forward 其实能并行处理很长的序列（Prefill 就是证据）。既然算力有余，能不能把「串行确认」变成「批量验证」——先大胆猜几个，再一次检查全对不对？这就是投机解码的出发点。

## 2. 投机解码的基本流程

具体流程分三步：

1. **Draft（猜）**：让一个便宜的小模型（Draft）连续生成 K 个候选 Token——小模型每步便宜得多，串行猜 4 个的成本可能还不如大模型一步
2. **Verify（验）**：把 K 个候选拼在一起，让大模型（Target）做**一次**前向——一次就能并行得到所有候选位置的概率，这正是序列并行计算的长处
3. **Accept（收）**：从左到右逐个检查候选。接受最长的一段正确前缀；第一个不对的位置，当场修正

三个角色里最反直觉的是「大模型一次 forward 怎么验证多个位置」——回忆 Attention 的因果掩蔽：位置 i 的输出只依赖位置 < i 的输入。把 K 个候选一起送进去，第 1 个位置的输出可以用来检查候选 1，第 2 个位置的输出检查候选 2……一次前向，K 份检查同时完成。

真正的难点不在「猜」，而在最后一步：**检查完怎么接受、怎么修正，才能让最终输出和直接用大模型采样完全相同？** 这是投机解码设计的精髓，接下来两节专门回答。

## 3. 接受与校正规则

设 Draft 对某候选 Token 给的概率是 $q(x)$，Target 给的是 $p(x)$。经典 speculative sampling 的接受概率是：

$$
a(x) = \min\left(1,\ \frac{p(x)}{q(x)}\right)
$$

直觉上读这个公式：Target 比 Draft 更看好的 Token（$p > q$），比值超过 1，直接接受；Target 不如 Draft 看好的（$p < q$），按比例接受——Target 的概率被 Draft 的概率「除」了一下，Draft 越自信、Target 越不看好，越容易被拒。

注意两个容易想错的点。第一，接受是**概率性的**，不是「Target 的 Top-1 和 Draft 相同才接受」——只要 $p(x) > 0$ 就有机会接受，哪怕它不是 Target 的第一名。第二，被拒绝之后**不能**直接从 Target 的分布重新采样。直接重采会让最终分布偏离 $p$——因为被拒绝的 Token 已经带着条件信息了。

正确的做法是从**校正分布**采样：$p'(x) = \mathrm{norm}(\max(p(x) - q(x), 0))$——只保留 Target 比 Draft 多出来的概率质量。可以证明（下一节的实验也会验证）：接受 + 校正组合起来，每个位置最终落在 $x$ 的概率恰好等于 $p(x)$。整套流程快是快了，但输出分布和直接采样 Target 一字不差——**这就是「投机」二字不亏心的原因**。

In [ ]:
draft_probs  = [0.80, 0.70, 0.60, 0.50]
target_probs = [0.90, 0.80, 0.20, 0.10]

for i,(q,p) in enumerate(zip(draft_probs,target_probs)):
    accept = min(1.0, p/q)
    print(f"pos {i}: target/draft={p/q:.2f}, accept_prob={accept:.2f}")


## 4. 投机解码的完整实现

把上面的规则组装成能跑的完整循环。模型用最便宜的玩具版：Target 和 Draft 都是「看到上一个 Token，给出下一个 Token 分布」的小表；Draft 的分布是 Target 的带噪声模仿——方向大体对、细节不准，就像真实场景里的小模型。

一轮做四件事：

```text
1. Draft 连续猜 gamma 个 Token
2. Target 一次 forward，并行得到所有候选位置的概率
3. 从左到右逐个检查：按 min(1, p/q) 的概率接受
4. 第一个拒绝的位置：从校正分布 max(p - q, 0) 归一化后采样一个 Token，本轮结束
   （全部接受时，再从 Target 直接采样一个「附赠」Token）
```

最后那个「附赠」值得多说一句：全对的时候，Target 这次 forward 顺便已经算出了下一个位置的概率——再白拿一个 Token，不拿白不拿。

In [ ]:
import numpy as np

vocab = ["今", "天", "气", "很", "好", "冷"]
idx = {t: i for i, t in enumerate(vocab)}

# 两张「当前 token -> 下一个 token 分数」的表：Target 更尖锐，Draft 是带噪声的模仿
rng = np.random.default_rng(0)
score_target = rng.normal(0, 2.0, (len(vocab), len(vocab)))
score_draft = score_target + rng.normal(0, 1.5, score_target.shape)

def dist(scores, token):
    """把当前 token 那一行分数变成下一个 token 的概率分布"""
    z = scores[idx[token]]
    e = np.exp(z - z.max())
    return e / e.sum()

def sample_from(p):
    return vocab[int(np.random.choice(len(p), p=p))]

def speculative_step(context_token, gamma=4):
    """一轮投机解码：返回这轮一共确认的 token 列表（只花 1 次 target forward）"""
    draft_tokens = []
    cur = context_token
    for _ in range(gamma):
        nxt = sample_from(dist(score_draft, cur))
        draft_tokens.append(nxt)
        cur = nxt

    accepted = []
    cur = context_token
    for t in draft_tokens:
        p = dist(score_target, cur)   # target 对这个位置的看法
        q = dist(score_draft, cur)    # draft 当时的依据
        if np.random.random() < min(1.0, p[idx[t]] / q[idx[t]]):
            accepted.append(t)
            cur = t
        else:
            residual = np.maximum(p - q, 0)   # 校正分布：target 多出来的概率质量
            accepted.append(sample_from(residual / residual.sum()))
            break
    if len(accepted) == len(draft_tokens):
        accepted.append(sample_from(dist(score_target, cur)))   # 附赠 token
    return accepted

np.random.seed(42)
print("一轮样例：", speculative_step("今", gamma=4))
print("再来一轮：", speculative_step("今", gamma=4))

In [ ]:
# 跑 2000 轮，统计「一次 target forward 平均确认几个 token」
np.random.seed(42)
rounds = 2000
accepted_counts = np.array([len(speculative_step("今", gamma=4)) for _ in range(rounds)])

print("每轮确认 token 数的均值:", round(accepted_counts.mean(), 3))
print("分布:", {k: int((accepted_counts == k).sum()) for k in range(1, 6)})
print("注意分布里没有 4：4 个 draft 全被接受时会附赠 1 个，直接变成 5")
print()
print("关键观察：普通 Decode 一次 forward 只确认 1 个 token；")
print("投机 Decode 一次 target forward + 4 次便宜的 draft，平均确认",
      round(accepted_counts.mean(), 2), "个")

In [ ]:
# 接受长度分布：大多数轮次确认 1-3 个，全接受（5 个）是少数
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.2))
plt.hist(accepted_counts, bins=np.arange(0.5, 6.5, 1),
         edgecolor="black", color="tab:blue")
plt.xticks(range(1, 6))
plt.xlabel("tokens confirmed per target forward")
plt.ylabel("rounds")
plt.title("Acceptance length distribution (gamma = 4)")
plt.show()

## 5. 加速比分析

先看实验里的数字：一轮（1 次 Target + 4 次便宜的 Draft）平均确认约 2.6 个 Token。粗略的加速比公式：

$$
\text{speedup} \approx \frac{\text{每轮平均确认的 Token 数}}{1 + \gamma \times \text{draft 单步成本（以 target 计）}}
$$

三个因素互相牵制。接受率越高越划算，这取决于 Draft 和 Target 有多「像」；猜的个数 $\gamma$ 不是越大越好——猜 8 个只接受 2 个，多出的 6 次 Draft 全是白算；Draft 也不是越小越好——太小猜得太离谱，接受率崩掉，Target 的 forward 反而更频繁。

所以真实系统里的投机解码是个调参问题：Draft 选多大、$\gamma$ 设几个、对什么负载开启（重复性高的任务接受率高，开放创作低），都要对着接受率数据调。

In [ ]:
def toy_speedup(k, accept_rate, draft_cost_ratio):
    # 一个教学用 proxy：每轮期望拿到 1 + 接受的 draft tokens
    expected_tokens = 1 + k * accept_rate
    cost = 1 + k * draft_cost_ratio
    return expected_tokens / cost

for a in [0.3,0.6,0.9]:
    print("accept", a, "proxy speedup", round(toy_speedup(4,a,0.08),2))


## 6. 分布一致性实验

本章第 3 节声称「接受 + 校正 = 输出分布和直接采样 Target 相同」。空口无凭，用 Monte Carlo 验证：构造一个两 Token 的分布，Target 是 `[0.7, 0.3]`，故意让 Draft 不同（`[0.5, 0.5]`），跑五万轮投机采样，数一数最终 Token 的频率。

如果理论正确，频率应该收敛到 Target 的 `[0.7, 0.3]`——Draft 的偏好被校正分布彻底「洗掉」了。

In [ ]:
import random, collections

p = [0.7, 0.3]  # target
q = [0.5, 0.5]  # draft

def sample(dist):
    r = random.random()
    return 0 if r < dist[0] else 1

def speculative_one():
    x = sample(q)
    if random.random() < min(1.0, p[x] / q[x]):
        return x
    residual = [max(p[i]-q[i],0.0) for i in range(2)]
    s = sum(residual)
    if s == 0:
        return sample(p)
    residual = [v/s for v in residual]
    return sample(residual)

random.seed(42)
n=50000
c=collections.Counter(speculative_one() for _ in range(n))
print("target:", p)
print("speculative empirical:", [round(c[i]/n,3) for i in range(2)])


实验结果就是投机解码的身份证：不管 Draft 的分布长什么样，最终频率都收敛到 Target。提速是过程，分布不变是底线——只提速度不保分布的任何「优化」，都不是投机解码。

## 7. 投机解码的变体

厂商报告里还有一串「同一家族」的名字，它们改变的是「谁来提案」，验证的骨架不变：

| 名词 | 怎么提案 |
|:---|:---|
| Speculative Decoding | 独立的小 Draft 模型 |
| Self-Speculative | 不另放小模型，用同一个模型的浅层 / 子网络提案 |
| Medusa | 给 Target 加几个额外的 head，每个 head 直接预测未来某个位置 |
| EAGLE | 在特征层面预测下一个 Token 的表示，再解码成 Token |
| Multi-Token Prediction | 训练时就让模型学会一次输出多个未来位置 |

看到任何一个新名字，先问那三个老问题：**谁在提案？谁在验证？一次 Target forward 平均确认几个 Token？** 名字会继续出新的，骨架就这一个。

## 小结

- 自回归的瓶颈是串行：每个 Token 都要等上一个确认；算力其实有余
- 投机解码让便宜的 Draft 先猜 $\gamma$ 个，Target 一次 forward 并行验证
- 接受准则按 min(1, p/q) 概率接受——不是「Top-1 相同才接受」
- 拒绝后从校正分布 norm(max(p−q, 0)) 采样，保证最终分布与 Target 相同
- 加速比由接受率、$\gamma$、Draft 开销共同决定；Draft 不是越小越好
- Medusa / EAGLE / MTP 换的是提案器，骨架不变

下一章把视角从「一个请求」切换到「一百个请求」：

> 单请求已经优化到位，但线上同时涌进来 100 个请求——谁先算、KV Cache 放哪、长 Prompt 和 Decode 怎么抢资源？

## 作业

三道题围绕三个核心量：校正分布、期望接受长度、加速比。

> **关于 AI 辅助**：可以让 AI 提示思路、拆解步骤，但不建议直接让 AI 完成题目。
> 这三笔账是理解投机解码全部收益与代价的钥匙。

### 作业 1：实现校正分布

Draft 被拒绝后，只能从 `max(p - q, 0)` 归一化后的分布里采样——只保留 Target 比 Draft
更偏好的概率质量。

**小提示**：`np.maximum(p - q, 0)` 之后再除以总和。

In [ ]:
# 作业 1：校正分布 填空

import numpy as np

p = np.array([0.5, 0.3, 0.2])   # target
q = np.array([0.6, 0.2, 0.2])   # draft

def correction_dist(p, q):
    """返回拒绝后应从中采样的校正分布"""
    residual = np.zeros_like(p)
    # TODO：把下面三引号里的内容替换成你的代码
    """residual 取 max(p - q, 0)，再归一化"""
    return residual

corr = correction_dist(p, q)
assert abs(corr.sum() - 1.0) < 1e-9, corr
assert corr[0] == 0.0, corr
print("✅ 作业 1 通过：校正分布只保留 target 多出来的概率质量")

### 作业 2：期望接受几个 Token

每个候选位置的条件接受概率是 $a_1, a_2, a_3$，期望接受的 draft token 数是
$a_1 + a_1 a_2 + a_1 a_2 a_3$——第 k 个被接受的前提是前面全部被接受。

**小提示**：一边累乘一边累加。

In [ ]:
# 作业 2：期望接受长度 填空

def expected_accepted(accept_probs):
    """给定每个位置的条件接受概率，返回期望接受的 token 数"""
    # TODO：把下面三引号里的内容替换成你的代码
    """按 a1 + a1*a2 + a1*a2*a3 + ... 累乘累加"""

assert abs(expected_accepted([0.9, 0.8, 0.7]) - 2.124) < 1e-9
assert expected_accepted([1.0, 1.0, 1.0]) == 3.0
print("✅ 作业 2 通过：你已经能量化「一轮能确认几个 Token」")

### 作业 3：这笔买卖划不划算

一轮的成本 = 1 次 Target forward + gamma 次 Draft forward。设 Draft 一次成本是 Target 的
`draft_cost`，每轮平均确认 `avg_tokens` 个 Token，则加速比
= `avg_tokens / (1 + gamma * draft_cost)`。

**小提示**：直接代入公式，和 1.0 比较。

In [ ]:
# 作业 3：加速比 填空

def speedup(avg_tokens, gamma, draft_cost):
    """返回相对普通 decode 的加速比"""
    # TODO：把下面三引号里的内容替换成你的代码
    """每轮成本 = 1 + gamma * draft_cost 次 target forward 当量"""

assert speedup(3.0, 4, 0.1) > 1.5
assert speedup(1.5, 8, 0.2) < 1.0
print("✅ 作业 3 通过：加速不是免费的，接受率和 draft 开销要一起算")